# 05 · Feature Extraction

**First heavy-compute notebook. Reads the fold CSVs from notebook 04.**

Extracts frozen ImageNet backbone features for both corpora and caches them to
`.npy`, so notebooks 06 and 07 train on arrays rather than pixels and a full
re-run costs minutes.

## Frozen, on purpose
The backbone is never fine-tuned here. Every downstream number is a linear probe
on ImageNet features, which makes the reported AUROC a floor rather than a tuned
ceiling. End-to-end fine-tuning is a separate experiment (notebook 07's Colab
companion).

## Three things that are load-bearing
- **Device**: cuda then mps then cpu. Missing the MPS branch once turned a
  five-minute job into forty-five.
- **Thread loader**: DataLoader workers spawn processes that cannot unpickle a
  notebook-defined class on macOS. Threads avoid it and still parallelise decode.
- **Row order on resume**: features align to labels by position, so a resumed
  run must return rows in input order. The extractor only appends, and asserts
  the final count.

## Backbones
EfficientNet-B0 is primary and runs on both tasks. ResNet50, MobileNetV2 and
DenseNet121 run on infection only, for the comparison in notebook 07. Each width
is asserted at build time, because head removal differs per family and a wrong
width fails silently.

## This notebook has a Colab companion
`05_feature_extraction_colab.ipynb` mounts Drive, stages images to local disk,
runs on a T4, and checkpoints. Use the Colab version to execute; this local
version documents the logic for the reproducibility package.

## Outputs
`feat_{backbone}_{task}.npy`, plus `index_{task}.csv` giving row order, fold, and
label.


In [ ]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, os

INTERIM = Path('data/interim')
FEATURES = Path('data/features'); FEATURES.mkdir(parents=True, exist_ok=True)

SEV_FOLDS = INTERIM / 'folds_severity.csv'
INF_FOLDS = INTERIM / 'folds_infection.csv'
RAW       = INTERIM / 'labels_raw.csv'

INPUT_SIZE = 224
BATCH      = 64
SEED       = 42

# backbones and their pooled feature widths, verified not assumed
BACKBONES = {
    'efficientnet_b0': ('IMAGENET1K_V1', 1280),
    'resnet50':        ('IMAGENET1K_V2', 2048),
    'mobilenet_v2':    ('IMAGENET1K_V2', 1280),
    'densenet121':     ('IMAGENET1K_V1', 1024),
}

missing = [str(p) for p in [SEV_FOLDS, INF_FOLDS, RAW] if not p.exists()]
if missing:
    print('STOPPING. Missing inputs:'); [print('  -', m) for m in missing]
    print('Run notebooks 01-04 first.')
    raise SystemExit(1)
print('inputs found.')

In [ ]:
# Cell 2 · device selection
# cuda > mps > cpu. The first version of this project checked only for CUDA,
# fell back to CPU on an Apple Silicon Mac, and took 45 minutes for what
# MPS does in five.
import torch
def pick_device():
    if torch.cuda.is_available(): return 'cuda'
    if torch.backends.mps.is_available(): return 'mps'
    return 'cpu'

DEV = pick_device()
torch.set_num_threads(os.cpu_count() or 4)
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'device: {DEV}   cores: {os.cpu_count()}')
if DEV == 'cpu':
    print('  no GPU backend. On Apple Silicon check torch is the arm64 build.')

In [ ]:
# Cell 3 · backbone builder, head removed
import torch.nn as nn, torchvision

def build_backbone(name):
    weights, expect_dim = BACKBONES[name]
    net = getattr(torchvision.models, name)(weights=weights)
    # head removal differs per family; getting it wrong silently yields
    # the wrong feature width
    if name.startswith('resnet'):
        net.fc = nn.Identity()
    else:                      # efficientnet, mobilenet, densenet
        net.classifier = nn.Identity()
    net.eval()
    # verify the width rather than trusting it
    with torch.no_grad():
        d = net(torch.randn(2, 3, INPUT_SIZE, INPUT_SIZE)).shape[1]
    assert d == expect_dim, f'{name}: got {d}, expected {expect_dim}'
    return net, d

print('backbone builder ready. widths will be asserted at build time.')

In [ ]:
# Cell 4 · image loader (threads, not DataLoader workers)
# DataLoader workers use 'spawn' on macOS and cannot unpickle a Dataset
# class defined in a notebook (it lives in __main__). Threads sidestep
# that entirely, and PIL releases the GIL during JPEG decode so decoding
# still runs in parallel.
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import torch

_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def load_one(path):
    try:
        with Image.open(path) as im:
            im = im.convert('RGB').resize((INPUT_SIZE, INPUT_SIZE))
        a = torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)
        return (a.permute(2, 0, 1) - _MEAN) / _STD
    except Exception:
        return torch.zeros(3, INPUT_SIZE, INPUT_SIZE)

def load_batch(paths, pool):
    return torch.stack(list(pool.map(load_one, paths)))

print('thread loader ready')

In [ ]:
# Cell 5 · extraction with resume, preserving row order
# Features are matched to labels BY POSITION downstream, so a resume must
# return rows in exactly the input order. We checkpoint the running array
# and its row count, and only ever append.
import torch, time

def extract(name, paths, tag):
    out_feat = FEATURES / f'feat_{name}_{tag}.npy'
    if out_feat.exists():
        arr = np.load(out_feat)
        if len(arr) == len(paths):
            print(f'  {name}/{tag}: cached ({len(arr):,} rows)')
            return arr
        print(f'  {name}/{tag}: partial ({len(arr):,}/{len(paths):,}), resuming')
        done = len(arr)
    else:
        arr, done = None, 0

    net, dim = build_backbone(name)
    net = net.to(DEV)
    feats = [] if arr is None else [arr]
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=8) as pool, torch.no_grad():
        for i in range(done, len(paths), BATCH):
            batch = load_batch(paths[i:i+BATCH], pool).to(DEV)
            f = net(batch).float().cpu().numpy()
            feats.append(f)
            if (i // BATCH) % 20 == 0:
                np.save(out_feat, np.concatenate(feats))   # checkpoint
                el = time.time() - t0
                print(f'    {i+len(f):,}/{len(paths):,}  ({el:.0f}s)')
    result = np.concatenate(feats)
    assert len(result) == len(paths), 'row count mismatch after extraction'
    np.save(out_feat, result)
    print(f'  {name}/{tag}: {result.shape} in {(time.time()-t0)/60:.1f} min')
    return result

print('extractor ready')

In [ ]:
# Cell 6 · run extraction for both corpora
# Primary backbone (EfficientNet-B0) on both tasks. The other three are
# extracted on the infection task only, for the backbone comparison in
# notebook 07. Order of rows follows the fold CSVs exactly.
sev = pd.read_csv(SEV_FOLDS)
inf = pd.read_csv(INF_FOLDS)

# severity uses one representative image per photograph
sev_paths = sev.representative.tolist()
# infection scores per image, then averages per unit in notebook 07
inf_paths = inf.path.tolist()

print(f'severity : {len(sev_paths):,} photographs')
print(f'infection: {len(inf_paths):,} images')

PRIMARY = 'efficientnet_b0'
print(f'\n[{PRIMARY}] severity')
feat_sev = extract(PRIMARY, sev_paths, 'severity')
print(f'[{PRIMARY}] infection')
feat_inf = extract(PRIMARY, inf_paths, 'infection')

# save row-order index so downstream can align without re-reading images
sev[['photo_unit', 'fold', 'y']].to_csv(FEATURES / 'index_severity.csv', index=False)
inf[['hash_cluster', 'photo_unit', 'fold', 'label', 'patient_id']].to_csv(
    FEATURES / 'index_infection.csv', index=False)
print('\nwrote row-order index files')

In [ ]:
# Cell 7 · other backbones on infection (for the comparison in nb 07)
# These are optional; skip if you only need EfficientNet-B0. Each is
# cached independently, so this cell is safe to interrupt and resume.
EXTRA = ['resnet50', 'mobilenet_v2', 'densenet121']
for name in EXTRA:
    print(f'[{name}] infection')
    try:
        extract(name, inf_paths, 'infection')
    except AssertionError as e:
        print(f'  skipped: {e}')
print('\nbackbone extraction complete')

In [ ]:
# Cell 8 · summary
print('=' * 58)
print('STAGE 05 COMPLETE')
print('=' * 58)
for f in sorted(FEATURES.glob('feat_*.npy')):
    arr = np.load(f, mmap_mode='r')
    print(f'  {f.name:<34} {arr.shape}')
print(f'\n  features cached in {FEATURES.resolve()}')
print('  downstream notebooks load these, not images.')
print('\nnext: 06_severity_models.ipynb')